In [1]:
import warnings
warnings.filterwarnings('ignore')

(Tutorial_Get_water_sites)=
# get_water_sites

*Recognizing neutral water molecules with indexed hydrogens.*

Recognize a covalent O-H-H graph independently of residue names or hydrogen-bond geometry. No implicit hydrogens are invented. The chemical tool can serve solvent analyses beyond interaction detection.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

See {func}`molsysmt.physchem.get_water_sites` for arguments, outputs and explicit failures.
:::

## Basic usage

Rows contain oxygen followed by two sorted hydrogen indices. All three atoms must be explicitly indexed.

In [2]:
import molsysmt as msm
from rdkit import Chem
molsys = Chem.AddHs(Chem.MolFromSmiles('O.O.CO'))
sites = msm.physchem.get_water_sites(molsys)
print(sites['water_atom_indices'])

[[0 4 5]
 [1 6 7]]


## Distinguishing chemical components

Hydroxide, hydronium, peroxide and alcohol are excluded. Oxygen-only water with implicit H is also excluded; no preparation or coordinates are generated.

In [3]:
for smiles in ['O', '[OH-]', '[OH3+]', 'CO', 'OO']:
    molsys = Chem.AddHs(Chem.MolFromSmiles(smiles))
    print(smiles, msm.physchem.get_water_sites(molsys)['water_atom_indices'].shape)
print('Implicit H:', msm.physchem.get_water_sites(Chem.MolFromSmiles('O'))['water_atom_indices'].shape)

O (1, 3)


[OH-] (0, 3)


[OH3+] (0, 3)
CO (0, 3)


OO (0, 3)
Implicit H: (0, 3)


## Selecting complete molecules

Recognition uses the full chemical graph before selection. A partial atom selection returns no water; isotope hydrogens retain source indices.

In [4]:
molsys = Chem.AddHs(Chem.MolFromSmiles('O'))
print(msm.physchem.get_water_sites(molsys, selection=[0,1])['water_atom_indices'])
print(msm.physchem.get_water_sites(molsys, selection=[2,0,1])['water_atom_indices'])
print(msm.physchem.get_water_sites(Chem.MolFromSmiles('[2H]O[2H]'))['water_atom_indices'])

[]


[[0 1 2]]


[[1 0 2]]


## Inspecting hydration components

Water coordinated by a metal remains a water in the covalent graph; a declared dative link is independent. Water virtual sites or other models require a separately defined profile. This recognition does not certify a force field.

In [5]:
molsys = Chem.AddHs(Chem.MolFromSmiles('[Zn+2].O.O.CC(=O)N'))
print(msm.physchem.get_water_sites(molsys)['water_atom_indices'])

[[ 1  7  8]
 [ 2  9 10]]


## Using native forms and chemical states

The same chemical tool accepts native MolSys and explicit reference-state indices. Coordinates are unnecessary for graph recognition.

In [6]:
molsys = msm.convert(molsys, to_form='molsysmt.MolSys')
sites = msm.physchem.get_water_sites(molsys, chemical_state=0, max_matches=1000)
print(sites['chemical_state_index'], sites['evidence'], sites['software'])
print(sites['attribution']['items'])

0 {'kind': 'declared_chemical_graph', 'connectivity_completeness': 'complete', 'assume_complete_connectivity': False, 'chemical_state_provenance_index': None} {'molsysmt': '0.21.0+606.ga03eb4bf6', 'rdkit': '2025.09.5'}
[{'id': 'software:molsysmt:0.21.0+606.ga03eb4bf6', 'type': 'software', 'title': 'MolSysMT', 'authors': ['Prada-Gracia, Diego', 'Moreno-Vargas, Liliana M.'], 'doi': '10.5281/zenodo.1298752', 'url': 'https://github.com/uibcdf/molsysmt', 'version': '0.21.0+606.ga03eb4bf6', 'roles': ['executed_software']}, {'id': 'software:rdkit:2025.09.5', 'type': 'software', 'title': 'RDKit: Open-source cheminformatics', 'url': 'https://www.rdkit.org', 'how_to_cite': 'https://www.rdkit.org/docs/Overview.html#citing-the-rdkit', 'version': '2025.09.5', 'roles': ['executed_software']}]


:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Sparse interaction results <user-tools-interactions-result>`
- {func}`molsysmt.Interactions.query` — querying observations by actual atom and structure indices.
:::